In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers datasets peft accelerate sentencepiece scikit-learn

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    "/content/drive/MyDrive/SHL_take_home_assessment"
)

DATA_PATH = Path(
    PROJECT_DIR
    / "version_4/data/source/multitask_training_dataset_v3.pkl"
)

df = pd.read_pickle(DATA_PATH)

train_df = df[df["split"] == "train"].copy()
validation_df = df[df["split"] == "validation"].copy()

print("Complete:", df.shape)
print("training:", train_df.shape)
print("Validation:", validation_df.shape)

Complete: (4000, 44)
training: (3200, 44)
Validation: (800, 44)


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

In [ ]:
def get_token_length(row):
    encoding = tokenizer(
        str(row["questionStatement"]),
        str(row["text"]),
        add_special_tokens = True,
        truncation = False,
    )

    return len(encoding["input_ids"])

df["token_length"] = df.apply(
    get_token_length,
    axis = 1,
)

print(df["token_length"].describe(
    percentiles = [0.90, 0.95, 0.99]
))

print(
    "Rows longer than 256:",
    (df["token_length"] > 256).sum(),
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (552 > 512). Running this sequence through the model will result in indexing errors


count    4000.000000
mean      200.346750
std        87.902501
min        34.000000
50%       174.000000
90%       344.000000
95%       368.000000
99%       413.000000
max       552.000000
Name: token_length, dtype: float64
Rows longer than 256: 1013


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA Available: True
GPU: Tesla T4


In [ ]:

PROJECT_DIR = Path(
    "/content/drive/MyDrive/SHL_take_home_assessment"
)

DATA_PATH = Path(
    PROJECT_DIR
    / "version_4/data/source/multitask_training_dataset_v3.pkl"
)


df = pd.read_pickle(DATA_PATH)

train_df = df[
    df["split"] == "train"
].copy()

validation_df = df[
    df["split"] == "validation"
].copy()

print("Complete:", df.shape)
print("Training:", train_df.shape)
print("Validation:", validation_df.shape)

assert len(train_df) == 3200
assert len(validation_df) == 800

Complete: (4000, 44)
Training: (3200, 44)
Validation: (800, 44)


In [ ]:
CEFR_TO_ID = {
    "A1": 0,
    "A2": 1,
    "B1": 2,
    "B2": 3,
    "C1": 4,
    "C2": 5,
}

ID_TO_CEFR = {
    value: key
    for key, value in CEFR_TO_ID.items()
}


def prepare_labels(frame):
    frame = frame.copy()

    frame["grammar_labels"] = (
        pd.to_numeric(
            frame["grammar"],
            errors="coerce",
        )
        .fillna(-100)
        .astype(float)
    )

    frame["content_labels"] = (
        pd.to_numeric(
            frame["content"],
            errors="coerce",
        )
        .fillna(-100)
        .astype(float)
    )

    # Exact CEFR label, used for evaluation.
    frame["cefr_labels"] = (
        frame["cefr"]
        .map(CEFR_TO_ID)
        .fillna(-100)
        .astype(int)
    )

    # Exact or interval endpoints, used for training.
    frame["cefr_min_labels"] = (
        pd.to_numeric(
            frame["cefr_min"],
            errors="coerce",
        )
        .fillna(-100)
        .astype(int)
    )

    frame["cefr_max_labels"] = (
        pd.to_numeric(
            frame["cefr_max"],
            errors="coerce",
        )
        .fillna(-100)
        .astype(int)
    )

    return frame


train_df = prepare_labels(train_df)
validation_df = prepare_labels(
    validation_df
)

print(
    "Exact CEFR training rows:",
    (train_df["cefr_labels"] != -100).sum(),
)

print(
    "All CEFR-supervised training rows:",
    (train_df["cefr_min_labels"] != -100).sum(),
)

print(
    "Interval CEFR training rows:",
    (
        (
            train_df["cefr_min_labels"] != -100
        )
        &
        (
            train_df["cefr_min_labels"]
            != train_df["cefr_max_labels"]
        )
    ).sum(),
)

Exact CEFR training rows: 1724
All CEFR-supervised training rows: 2080
Interval CEFR training rows: 356


In [ ]:
assert train_df[
    "grammar_labels"
].isin([-100, 0, 1, 2, 3, 4, 5]).all()

assert train_df[
    "content_labels"
].isin([-100, 0, 1, 2, 3, 4]).all()

assert train_df[
    "cefr_labels"
].isin([-100, 0, 1, 2, 3, 4, 5]).all()

valid_intervals = (
    train_df["cefr_min_labels"] >= 0
)

assert (
    train_df.loc[
        valid_intervals,
        "cefr_min_labels",
    ]
    <=
    train_df.loc[
        valid_intervals,
        "cefr_max_labels",
    ]
).all()

In [ ]:
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 384

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast = True,
)

print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocabulary size:", len(tokenizer))
print("Padding token:", tokenizer.pad_token)

Tokenizer: XLMRobertaTokenizer
Vocabulary size: 250002
Padding token: <pad>


In [ ]:
MODEL_COLUMNS = [
    "questionStatement",
    "text",
    "grammar_labels",
    "content_labels",
    "cefr_labels",
    "cefr_min_labels",
    "cefr_max_labels",
]
train_dataset = Dataset.from_pandas(
    train_df[MODEL_COLUMNS],
    preserve_index = False,
)

validation_dataset = Dataset.from_pandas(
    validation_df[MODEL_COLUMNS],
    preserve_index = False,
)

raw_datasets = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset,
})

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['questionStatement', 'text', 'grammar_labels', 'content_labels', 'cefr_labels', 'cefr_min_labels', 'cefr_max_labels'],
        num_rows: 3200
    })
    validation: Dataset({
        features: ['questionStatement', 'text', 'grammar_labels', 'content_labels', 'cefr_labels', 'cefr_min_labels', 'cefr_max_labels'],
        num_rows: 800
    })
})


In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["questionStatement"],
        batch["text"],
        truncation = True,
        max_length = MAX_LENGTH,
        padding = False,
    )

tokenized_datasets = raw_datasets.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "questionStatement",
        "text",
    ],
    desc="Tokenizing multilingual emails",
)

print(tokenized_datasets)
print(tokenized_datasets["train"].column_names)

## attention_mask tells XLM-R which positions contain real tokens and which are padding:

Tokenizing multilingual emails:   0%|          | 0/3200 [00:00<?, ? examples/s]

Tokenizing multilingual emails:   0%|          | 0/800 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['grammar_labels', 'content_labels', 'cefr_labels', 'cefr_min_labels', 'cefr_max_labels', 'input_ids', 'attention_mask'],
        num_rows: 3200
    })
    validation: Dataset({
        features: ['grammar_labels', 'content_labels', 'cefr_labels', 'cefr_min_labels', 'cefr_max_labels', 'input_ids', 'attention_mask'],
        num_rows: 800
    })
})
['grammar_labels', 'content_labels', 'cefr_labels', 'cefr_min_labels', 'cefr_max_labels', 'input_ids', 'attention_mask']


In [ ]:
train_lengths = [
    len(row["input_ids"])
    for row in tokenized_datasets["train"]
]

validation_lengths = [
    len(row["input_ids"])
    for row in tokenized_datasets["validation"]
]

print("Maximum training length:", max(train_lengths))
print("Maximum validation length:", max(validation_lengths))

assert max(train_lengths) <= MAX_LENGTH
assert max(validation_lengths) <= MAX_LENGTH

print("Tokenization validation: PASS")

Maximum training length: 384
Maximum validation length: 384
Tokenization validation: PASS


In [ ]:
class MultitaskDataCollator:

    def __init__(
        self,
        tokenizer,
        pad_to_multiple_of=8,
    ):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features):
        model_features = [
            {
                key: feature[key]
                for key in self.tokenizer.model_input_names
                if key in feature
            }
            for feature in features
        ]

        batch = self.tokenizer.pad(
            model_features,
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        for label_name in [
            "grammar_labels",
            "content_labels",
        ]:
            batch[label_name] = torch.tensor(
                [
                    feature[label_name]
                    for feature in features
                ],
                dtype=torch.float32,
            )

        for label_name in [
            "cefr_labels",
            "cefr_min_labels",
            "cefr_max_labels",
        ]:
            batch[label_name] = torch.tensor(
                [
                    feature[label_name]
                    for feature in features
                ],
                dtype=torch.long,
            )

        return batch

data_collator = MultitaskDataCollator(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)

In [ ]:
test_features = [
    tokenized_datasets["train"][index]
    for index in range(8)
]

test_batch = data_collator(test_features)

for key, value in test_batch.items():
    print(
        key,
        tuple(value.shape),
        value.dtype,
    )

input_ids (8, 120) torch.int64
attention_mask (8, 120) torch.int64
grammar_labels (8,) torch.float32
content_labels (8,) torch.float32
cefr_labels (8,) torch.int64
cefr_min_labels (8,) torch.int64
cefr_max_labels (8,) torch.int64


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModel
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)

In [ ]:
class OrderedCEFRHead(nn.Module):
    """
    Produces five ordered logits:

    P(level > A1)
    P(level > A2)
    P(level > B1)
    P(level > B2)
    P(level > C1)
    """

    def __init__(self, input_size):
        super().__init__()

        self.score = nn.Linear(
            input_size,
            1,
        )

        # Initial thresholds:
        # [-2, -1, 0, 1, 2]
        self.base_threshold = nn.Parameter(
            torch.tensor(-2.0)
        )

        inverse_softplus_one = math.log(
            math.exp(1.0) - 1.0
        )

        self.raw_steps = nn.Parameter(
            torch.full(
                (4,),
                inverse_softplus_one,
            )
        )

    def get_thresholds(self):
        positive_steps = F.softplus(
            self.raw_steps
        )

        remaining_thresholds = (
            self.base_threshold
            + torch.cumsum(
                positive_steps,
                dim = 0,
            )
        )

        return torch.cat([
            self.base_threshold.reshape(1),
            remaining_thresholds,
        ])

    def forward(self, hidden_states):
        score = self.score(hidden_states)
        thresholds = self.get_thresholds()
        return score - thresholds.unsqueeze(0)

In [ ]:
class XLMRMultitaskModel(nn.Module):

    def __init__(
        self,
        model_name="xlm-roberta-base",
        projection_size=256,
        dropout=0.10,
    ):
        super().__init__()

        base_encoder = AutoModel.from_pretrained(
            model_name
        )

        lora_config = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=8,
            lora_alpha=16,
            lora_dropout=0.10,
            target_modules=[
                "query",
                "value",
            ],
            bias="none",
        )

        self.encoder = get_peft_model(
            base_encoder,
            lora_config,
        )

        hidden_size = (
            base_encoder.config.hidden_size
        )

        self.shared_projection = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(
                hidden_size,
                projection_size,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.grammar_head = nn.Linear(
            projection_size,
            1,
        )

        self.content_head = nn.Linear(
            projection_size,
            1,
        )

        self.cefr_head = OrderedCEFRHead(
            projection_size
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        grammar_labels=None,
        content_labels=None,
        cefr_labels=None,
        cefr_min_labels=None,
        cefr_max_labels=None,
    ):
        encoder_output = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )


        token_embeddings = (
        encoder_output.last_hidden_state
        )

        expanded_mask = (
            attention_mask
            .unsqueeze(-1)
            .to(token_embeddings.dtype)
        )

        pooled_output = (
            (
                token_embeddings
                * expanded_mask
            ).sum(dim=1)
            / expanded_mask.sum(
                dim=1
            ).clamp(min=1e-9)
        )

        features = self.shared_projection(
            pooled_output
        )

        grammar_predictions = (
            torch.sigmoid(
                self.grammar_head(features)
            )
            .squeeze(-1)
            * 5.0
        )

        content_predictions = (
            torch.sigmoid(
                self.content_head(features)
            )
            .squeeze(-1)
            * 4.0
        )

        cefr_logits = self.cefr_head(
            features
        )

        # Number of ordinal thresholds passed:
        # 0=A1, ..., 5=C2.
        cefr_predictions = (
            torch.sigmoid(cefr_logits)
            > 0.5
        ).sum(dim=1)

        output = {
            "grammar_predictions":
                grammar_predictions,

            "content_predictions":
                content_predictions,

            "cefr_logits":
                cefr_logits,

            "cefr_predictions":
                cefr_predictions,
        }

        losses = []

        # -------------------------------------------------
        # Grammar loss
        # -------------------------------------------------
        if grammar_labels is not None:
            grammar_mask = (
                grammar_labels != -100
            )

            if grammar_mask.any():
                grammar_loss = F.mse_loss(
                    (
                        grammar_predictions[
                            grammar_mask
                        ]
                        / 5.0
                    ),
                    (
                        grammar_labels[
                            grammar_mask
                        ]
                        / 5.0
                    ),
                )

                output["grammar_loss"] = (
                    grammar_loss
                )

                losses.append(
                    grammar_loss
                )

        # -------------------------------------------------
        # Content loss
        # -------------------------------------------------
        if content_labels is not None:
            content_mask = (
                content_labels != -100
            )

            if content_mask.any():
                content_loss = F.mse_loss(
                    (
                        content_predictions[
                            content_mask
                        ]
                        / 4.0
                    ),
                    (
                        content_labels[
                            content_mask
                        ]
                        / 4.0
                    ),
                )

                output["content_loss"] = (
                    content_loss
                )

                losses.append(
                    content_loss
                )

        # -------------------------------------------------
        # Interval-aware ordinal CEFR loss
        # -------------------------------------------------
        if (
            cefr_min_labels is not None
            and cefr_max_labels is not None
        ):
            cefr_mask = (
                (cefr_min_labels >= 0)
                &
                (cefr_max_labels >= 0)
            )

            if cefr_mask.any():
                valid_logits = (
                    cefr_logits[cefr_mask]
                )

                minimum_levels = (
                    cefr_min_labels[
                        cefr_mask
                    ]
                )

                maximum_levels = (
                    cefr_max_labels[
                        cefr_mask
                    ]
                )

                thresholds = torch.arange(
                    5,
                    device=cefr_logits.device,
                ).unsqueeze(0)

                # Thresholds that every possible
                # label in the interval passes.
                known_positive = (
                    thresholds
                    < minimum_levels.unsqueeze(1)
                )

                # Thresholds that no possible
                # label in the interval passes.
                known_negative = (
                    thresholds
                    >= maximum_levels.unsqueeze(1)
                )

                known_thresholds = (
                    known_positive
                    | known_negative
                )

                ordinal_targets = (
                    known_positive.float()
                )

                cefr_loss_matrix = (
                    F.binary_cross_entropy_with_logits(
                        valid_logits,
                        ordinal_targets,
                        reduction="none",
                    )
                )

                cefr_loss = (
                    cefr_loss_matrix[
                        known_thresholds
                    ].mean()
                )

                output["cefr_loss"] = (
                    cefr_loss
                )

                losses.append(
                    cefr_loss
                )

        if losses:
            output["loss"] = torch.stack(
                losses
            ).sum()

        return output

In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = XLMRMultitaskModel(
    model_name=MODEL_NAME,
    projection_size=256,
    dropout=0.10,
)

model.to(device)

model.encoder.print_trainable_parameters()

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 294,912 || all params: 278,338,560 || trainable%: 0.1060


In [ ]:
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Total trainable:", trainable_parameters)
print("Total parameters:", total_parameters)
print(
    "Trainable percentage:",
    100 * trainable_parameters / total_parameters,
)

Total trainable: 494088
Total parameters: 278537736
Trainable percentage: 0.17738637754993455


In [ ]:
test_batch_device = {
    key: value.to(device)
    for key, value in test_batch.items()
}

model.eval()

with torch.no_grad():
    test_output = model(
        **test_batch_device
    )

print(
    "Loss:",
    test_output["loss"].item(),
)

print(
    "Grammar predictions:",
    test_output[
        "grammar_predictions"
    ][:5],
)

print(
    "Content predictions:",
    test_output[
        "content_predictions"
    ][:5],
)

print(
    "CEFR predictions:",
    test_output[
        "cefr_predictions"
    ][:5],
)

print(
    "Ordered thresholds:",
    model.cefr_head
    .get_thresholds()
    .detach()
    .cpu(),
)

Loss: 1.2453863620758057
Grammar predictions: tensor([2.6762, 2.6660, 2.6740, 2.6764, 2.6603], device='cuda:0')
Content predictions: tensor([2.2411, 2.2654, 2.2582, 2.2234, 2.2516], device='cuda:0')
CEFR predictions: tensor([2, 2, 2, 2, 2], device='cuda:0')
Ordered thresholds: tensor([-2., -1.,  0.,  1.,  2.])


#TRaining

In [ ]:
from torch.utils.data import DataLoader

TRAIN_BATCH_SIZE = 8
VALIDATION_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2

train_loader = DataLoader(
    tokenized_datasets["train"],
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=2,
)

validation_loader = DataLoader(
    tokenized_datasets["validation"],
    batch_size=VALIDATION_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=2,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print(
    "Effective batch size:",
    TRAIN_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS,
)

Training batches: 400
Validation batches: 50
Effective batch size: 16


In [ ]:
from scipy.stats import pearsonr
from sklearn.metrics import accuracy_score

import numpy as np
import torch


def safe_pearson(targets, predictions):
    targets = np.asarray(targets)
    predictions = np.asarray(predictions)

    if (
        len(targets) < 2
        or np.std(targets) == 0
        or np.std(predictions) == 0
    ):
        return 0.0

    return float(
        pearsonr(
            targets,
            predictions,
        ).statistic
    )


@torch.no_grad()
def evaluate_model(
    model,
    data_loader,
    device,
):
    model.eval()

    losses = []

    grammar_targets = []
    grammar_predictions = []

    content_targets = []
    content_predictions = []

    cefr_targets = []
    cefr_hard_predictions = []
    cefr_soft_predictions = []

    for batch in data_loader:
        batch = {
            key: value.to(
                device,
                non_blocking=True,
            )
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=device.type == "cuda",
        ):
            output = model(**batch)

        if "loss" in output:
            losses.append(
                output["loss"]
                .detach()
                .float()
                .item()
            )

        # -------------------------------------------------
        # Grammar metrics: only rows with grammar labels
        # -------------------------------------------------
        grammar_mask = (
            batch["grammar_labels"] != -100
        )

        if grammar_mask.any():
            grammar_targets.extend(
                batch["grammar_labels"][grammar_mask]
                .detach()
                .float()
                .cpu()
                .numpy()
            )

            grammar_predictions.extend(
                output["grammar_predictions"][grammar_mask]
                .detach()
                .float()
                .cpu()
                .numpy()
            )

        # -------------------------------------------------
        # Content metrics: only rows with content labels
        # -------------------------------------------------
        content_mask = (
            batch["content_labels"] != -100
        )

        if content_mask.any():
            content_targets.extend(
                batch["content_labels"][content_mask]
                .detach()
                .float()
                .cpu()
                .numpy()
            )

            content_predictions.extend(
                output["content_predictions"][content_mask]
                .detach()
                .float()
                .cpu()
                .numpy()
            )

        # -------------------------------------------------
        # CEFR metrics: only rows with exact CEFR labels
        #
        # Interval-labelled examples have cefr_labels=-100,
        # so they help training/loss but are not used for
        # exact-label accuracy or Pearson correlation.
        # -------------------------------------------------
        cefr_mask = (
            batch["cefr_labels"] != -100
        )

        if cefr_mask.any():
            valid_cefr_targets = (
                batch["cefr_labels"][cefr_mask]
            )

            valid_cefr_logits = (
                output["cefr_logits"][cefr_mask]
            )

            probabilities = torch.sigmoid(
                valid_cefr_logits
            )

            hard_predictions = (
                probabilities > 0.5
            ).sum(dim=1)

            # Continuous ordinal score in [0, 5].
            # Used only for diagnostic Pearson correlation.
            soft_predictions = probabilities.sum(
                dim=1
            )

            cefr_targets.extend(
                valid_cefr_targets
                .detach()
                .cpu()
                .numpy()
            )

            cefr_hard_predictions.extend(
                hard_predictions
                .detach()
                .cpu()
                .numpy()
            )

            cefr_soft_predictions.extend(
                soft_predictions
                .detach()
                .float()
                .cpu()
                .numpy()
            )

    metrics = {
        "validation_loss": (
            float(np.mean(losses))
            if losses
            else float("nan")
        ),

        "grammar_pearson": safe_pearson(
            grammar_targets,
            grammar_predictions,
        ),

        "content_pearson": safe_pearson(
            content_targets,
            content_predictions,
        ),

        "cefr_pearson_hard": safe_pearson(
            cefr_targets,
            cefr_hard_predictions,
        ),

        "cefr_pearson_soft": safe_pearson(
            cefr_targets,
            cefr_soft_predictions,
        ),

        "cefr_accuracy": (
            float(
                accuracy_score(
                    cefr_targets,
                    cefr_hard_predictions,
                )
            )
            if cefr_targets
            else float("nan")
        ),

        # Useful sanity-check counts
        "grammar_evaluation_rows":
            len(grammar_targets),

        "content_evaluation_rows":
            len(content_targets),

        "cefr_exact_evaluation_rows":
            len(cefr_targets),
    }

    return metrics

In [ ]:
import random
from transformers import (
    get_linear_schedule_with_warmup,
)

SEED = 42
EPOCHS = 8

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.set_float32_matmul_precision("high")

encoder_parameters = [
    parameter
    for name, parameter
    in model.named_parameters()
    if (
        parameter.requires_grad
        and name.startswith("encoder.")
    )
]

head_parameters = [
    parameter
    for name, parameter
    in model.named_parameters()
    if (
        parameter.requires_grad
        and not name.startswith("encoder.")
    )
]

optimizer = torch.optim.AdamW(
    [
        {
            "params": encoder_parameters,
            "lr": 2e-4,
            "weight_decay": 0.01,
        },
        {
            "params": head_parameters,
            "lr": 5e-4,
            "weight_decay": 0.01,
        },
    ]
)

updates_per_epoch = math.ceil(
    len(train_loader)
    / GRADIENT_ACCUMULATION_STEPS
)

total_training_steps = (
    updates_per_epoch * EPOCHS
)

warmup_steps = int(
    0.10 * total_training_steps
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=device.type == "cuda",
)

print("Updates per epoch:", updates_per_epoch)
print("Total optimizer updates:", total_training_steps)
print("Warmup updates:", warmup_steps)

Updates per epoch: 200
Total optimizer updates: 1600
Warmup updates: 160


In [ ]:
from tqdm.auto import tqdm

history = []
best_score = -float("inf")
best_epoch = None
best_trainable_state = None

optimizer.zero_grad(set_to_none=True)

for epoch in range(1, EPOCHS + 1):
    model.train()

    running_loss = 0.0

    progress_bar = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch}/{EPOCHS}",
    )

    for step, batch in progress_bar:
        batch = {
            key: value.to(
                device,
                non_blocking=True,
            )
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=device.type == "cuda",
        ):
            output = model(**batch)

            loss = (
                output["loss"]
                / GRADIENT_ACCUMULATION_STEPS
            )

        scaler.scale(loss).backward()

        should_update = (
            (step + 1)
            % GRADIENT_ACCUMULATION_STEPS
            == 0
            or (step + 1) == len(train_loader)
        )

        if should_update:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                [
                    parameter
                    for parameter
                    in model.parameters()
                    if parameter.requires_grad
                ],
                max_norm=1.0,
            )

            scaler.step(optimizer)
            scaler.update()

            scheduler.step()
            optimizer.zero_grad(
                set_to_none=True
            )

        running_loss += (
            loss.item()
            * GRADIENT_ACCUMULATION_STEPS
        )

        progress_bar.set_postfix(
            loss=(
                running_loss
                / (step + 1)
            ),
            lr=scheduler.get_last_lr()[0],
        )

    metrics = evaluate_model(
        model=model,
        data_loader=validation_loader,
        device=device,
    )

    metrics["epoch"] = epoch
    metrics["training_loss"] = (
        running_loss / len(train_loader)
    )

    # Composite checkpoint-selection score
    metrics["selection_score"] = np.mean([
        metrics["grammar_pearson"],
        metrics["content_pearson"],
        metrics["cefr_pearson_soft"],
        metrics["cefr_accuracy"],
    ])

    history.append(metrics)

    print("\nEpoch", epoch)
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

    if metrics["selection_score"] > best_score:
        best_score = metrics[
            "selection_score"
        ]

        best_epoch = epoch

        # Only LoRA and prediction-head weights
        best_trainable_state = {
            name: parameter
            .detach()
            .cpu()
            .clone()

            for name, parameter
            in model.named_parameters()

            if parameter.requires_grad
        }

        print(
            "Saved best in-memory checkpoint."
        )


Epoch 1/8:   0%|          | 0/400 [00:00<?, ?it/s]


Epoch 1
validation_loss: 0.4142
grammar_pearson: 0.4633
content_pearson: 0.9285
cefr_pearson_hard: 0.7287
cefr_pearson_soft: 0.7798
cefr_accuracy: 0.4488
grammar_evaluation_rows: 400
content_evaluation_rows: 400
cefr_exact_evaluation_rows: 430
epoch: 1
training_loss: 0.5735
selection_score: 0.6551
Saved best in-memory checkpoint.


Epoch 2/8:   0%|          | 0/400 [00:00<?, ?it/s]


Epoch 2
validation_loss: 0.3523
grammar_pearson: 0.6907
content_pearson: 0.9677
cefr_pearson_hard: 0.8075
cefr_pearson_soft: 0.8339
cefr_accuracy: 0.5070
grammar_evaluation_rows: 400
content_evaluation_rows: 400
cefr_exact_evaluation_rows: 430
epoch: 2
training_loss: 0.4096
selection_score: 0.7498
Saved best in-memory checkpoint.


Epoch 3/8:   0%|          | 0/400 [00:00<?, ?it/s]

...


Epoch 6
validation_loss: 0.2872
grammar_pearson: 0.9034
content_pearson: 0.9757
cefr_pearson_hard: 0.8438
cefr_pearson_soft: 0.8667
cefr_accuracy: 0.6047
grammar_evaluation_rows: 400
content_evaluation_rows: 400
cefr_exact_evaluation_rows: 430
epoch: 6
training_loss: 0.2799
selection_score: 0.8376


Epoch 7/8:   0%|          | 0/400 [00:00<?, ?it/s]


Epoch 7
validation_loss: 0.2891
grammar_pearson: 0.9065
content_pearson: 0.9756
cefr_pearson_hard: 0.8359
cefr_pearson_soft: 0.8588
cefr_accuracy: 0.6140
grammar_evaluation_rows: 400
content_evaluation_rows: 400
cefr_exact_evaluation_rows: 430
epoch: 7
training_loss: 0.2661
selection_score: 0.8387


Epoch 8/8:   0%|          | 0/400 [00:00<?, ?it/s]


Epoch 8
validation_loss: 0.2868
grammar_pearson: 0.9063
content_pearson: 0.9767
cefr_pearson_hard: 0.8354
cefr_pearson_soft: 0.8580
cefr_accuracy: 0.6256
grammar_evaluation_rows: 400
content_evaluation_rows: 400
cefr_exact_evaluation_rows: 430
epoch: 8
training_loss: 0.2613
selection_score: 0.8417
Saved best in-memory checkpoint.


In [ ]:
print(tokenized_datasets["validation"].column_names)

validation_batch = next(iter(validation_loader))
print(validation_batch.keys())

['grammar_labels', 'content_labels', 'cefr_labels', 'cefr_min_labels', 'cefr_max_labels', 'input_ids', 'attention_mask']
KeysView({'input_ids': tensor([[     0,    891,   3860,  ...,      1,      1,      1],
        [     0, 235444,    293,  ...,      1,      1,      1],
        [     0,   7806,  13478,  ...,      1,      1,      1],
        ...,
        [     0,  98719,   1330,  ...,      1,      1,      1],
        [     0,   6096,   4391,  ...,      1,      1,      1],
        [     0,    891,   3860,  ...,      1,      1,      1]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'cefr_labels': tensor([-100, -100, -100, -100, -100,    0, -100, -100, -100, -100,    0, -100,
        -100,    0, -100, -100]), 'cefr_min_labels': tensor([-100, -100, -100, -100, -100,    0, -100, -100, -100, -100,    0

In [ ]:
# Restore the best checkpoint
with torch.no_grad():
    for name, parameter in model.named_parameters():
        if name in best_trainable_state:
            parameter.copy_(
                best_trainable_state[name].to(
                    parameter.device
                )
            )

print("Restored best epoch:", best_epoch)
print("Best selection score:", best_score)

Restored best epoch: 8
Best selection score: 0.841661827213087


In [ ]:
import json

CHECKPOINT_DIR = (
    PROJECT_DIR
    / "version_5/models/xlmr_lora_multitask_mean_pool_epoch8"
)

ADAPTER_DIR = CHECKPOINT_DIR / "lora_adapter"
TOKENIZER_DIR = CHECKPOINT_DIR / "tokenizer"

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Save LoRA adapter
model.encoder.save_pretrained(
    str(ADAPTER_DIR)
)

# Save tokenizer
tokenizer.save_pretrained(
    str(TOKENIZER_DIR)
)


def cpu_state_dict(module):
    return {
        name: tensor.detach().cpu()
        for name, tensor
        in module.state_dict().items()
    }


# Save shared projection and prediction heads
torch.save(
    {
        "shared_projection":
            cpu_state_dict(
                model.shared_projection
            ),

        "grammar_head":
            cpu_state_dict(
                model.grammar_head
            ),

        "content_head":
            cpu_state_dict(
                model.content_head
            ),

        "cefr_head":
            cpu_state_dict(
                model.cefr_head
            ),
    },
    CHECKPOINT_DIR / "prediction_heads_epoch8.pt",
)

In [ ]:
required_paths = [
    ADAPTER_DIR / "adapter_config.json",
    ADAPTER_DIR / "adapter_model.safetensors",
    CHECKPOINT_DIR / "prediction_heads_epoch8.pt",
]

for path in required_paths:
    print(path, "->", path.exists())
    assert path.exists()

print("Checkpoint files: PASS")

/content/drive/MyDrive/SHL_take_home_assessment/version_5/models/xlmr_lora_multitask_mean_pool_epoch8/lora_adapter/adapter_config.json -> True
/content/drive/MyDrive/SHL_take_home_assessment/version_5/models/xlmr_lora_multitask_mean_pool_epoch8/lora_adapter/adapter_model.safetensors -> True
/content/drive/MyDrive/SHL_take_home_assessment/version_5/models/xlmr_lora_multitask_mean_pool_epoch8/prediction_heads_epoch8.pt -> True
Checkpoint files: PASS


In [ ]:
final_validation_metrics = evaluate_model(
    model=model,
    data_loader=validation_loader,
    device=device,
)

print(final_validation_metrics)

{'validation_loss': 0.2868427651003003, 'grammar_pearson': 0.906326413154602, 'content_pearson': 0.9767326712608337, 'cefr_pearson_hard': 0.8353739719332791, 'cefr_pearson_soft': 0.8580068290880751, 'cefr_accuracy': 0.6255813953488372, 'grammar_evaluation_rows': 400, 'content_evaluation_rows': 400, 'cefr_exact_evaluation_rows': 430}


In [ ]:
metadata = {
    "base_model": MODEL_NAME,
    "epochs": EPOCHS,
    "best_epoch": best_epoch,
    "best_selection_score": float(best_score),
    "max_length": MAX_LENGTH,
    "projection_size": 256,
    "lora_rank": 8,
    "lora_alpha": 16,
    "cefr_training": "exact_and_interval_ordinal",
    "grammar_missing_label": -100,
    "content_missing_label": -100,
    "cefr_missing_label": -100,
}

with open(
    CHECKPOINT_DIR / "training_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )

with open(
    CHECKPOINT_DIR / "training_history.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        history,
        file,
        indent=2,
        ensure_ascii=False,
        default=float,
    )

print("Metadata and history saved.")

Metadata and history saved.


In [ ]:
@torch.no_grad()
def collect_cefr_validation_scores(
    model,
    data_loader,
    device,
):
    model.eval()

    targets = []
    soft_scores = []

    for batch in data_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        output = model(**batch)

        # Only exact CEFR labels are valid for calibration
        mask = batch["cefr_labels"] != -100

        if mask.any():
            logits = output["cefr_logits"][mask]

            scores = torch.sigmoid(
                logits
            ).sum(dim=1)

            targets.extend(
                batch["cefr_labels"][mask]
                .detach()
                .cpu()
                .numpy()
            )

            soft_scores.extend(
                scores.detach()
                .float()
                .cpu()
                .numpy()
            )

    return (
        np.asarray(targets),
        np.asarray(soft_scores),
    )


cefr_targets, cefr_soft_scores = (
    collect_cefr_validation_scores(
        model=model,
        data_loader=validation_loader,
        device=device,
    )
)

print("Exact calibration rows:", len(cefr_targets))

Exact calibration rows: 430


In [ ]:
def find_adjacent_cut(
    targets,
    scores,
    lower_level,
    upper_level,
):
    mask = np.isin(
        targets,
        [lower_level, upper_level],
    )

    pair_targets = targets[mask]
    pair_scores = scores[mask]

    unique_scores = np.unique(pair_scores)

    candidates = (
        unique_scores[:-1]
        + unique_scores[1:]
    ) / 2.0

    best_cut = None
    best_accuracy = -1.0

    for cut in candidates:
        predictions = np.where(
            pair_scores > cut,
            upper_level,
            lower_level,
        )

        accuracy = accuracy_score(
            pair_targets,
            predictions,
        )

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_cut = float(cut)

    return best_cut, float(best_accuracy)


cut_points = []
pair_accuracies = []

for lower_level in range(5):
    upper_level = lower_level + 1

    cut, pair_accuracy = find_adjacent_cut(
        cefr_targets,
        cefr_soft_scores,
        lower_level,
        upper_level,
    )

    cut_points.append(cut)
    pair_accuracies.append(pair_accuracy)

    print(
        f"{ID_TO_CEFR[lower_level]} / "
        f"{ID_TO_CEFR[upper_level]}: "
        f"cut={cut:.4f}, "
        f"accuracy={pair_accuracy:.4f}"
    )

cut_points = np.asarray(cut_points)

assert np.all(np.diff(cut_points) > 0), (
    f"Calibration thresholds are not ordered: "
    f"{cut_points}"
)

A1 / A2: cut=0.6242, accuracy=0.8750
A2 / B1: cut=1.5729, accuracy=0.8228
B1 / B2: cut=2.2836, accuracy=0.7709
B2 / C1: cut=3.4687, accuracy=0.8455
C1 / C2: cut=4.5330, accuracy=0.8814


In [ ]:
calibrated_predictions = np.digitize(
    cefr_soft_scores,
    bins=cut_points,
)

calibration_metrics = {
    "exact_accuracy": float(
        accuracy_score(
            cefr_targets,
            calibrated_predictions,
        )
    ),
    "pearson": safe_pearson(
        cefr_targets,
        calibrated_predictions,
    ),
    "mean_distance": float(
        np.mean(
            np.abs(
                cefr_targets
                - calibrated_predictions
            )
        )
    ),
    "within_one_level": float(
        np.mean(
            np.abs(
                cefr_targets
                - calibrated_predictions
            ) <= 1
        )
    ),
}

calibration_data = {
    "method": "adjacent_class_validation_cutpoints",
    "cut_points": cut_points.tolist(),
    "pair_accuracies": pair_accuracies,
    "class_mapping": ID_TO_CEFR,
    "exact_validation_rows": int(
        len(cefr_targets)
    ),
    "metrics": calibration_metrics,
}

with open(
    CHECKPOINT_DIR / "cefr_calibration.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        calibration_data,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Cut points:", cut_points)
print("Metrics:", calibration_metrics)
print(
    "Saved:",
    CHECKPOINT_DIR / "cefr_calibration.json",
)

Cut points: [0.62415582 1.57292855 2.28356171 3.46867323 4.53301334]
Metrics: {'exact_accuracy': 0.6767441860465117, 'pearson': 0.8521710374023652, 'mean_distance': 0.3674418604651163, 'within_one_level': 0.958139534883721}
Saved: /content/drive/MyDrive/SHL_take_home_assessment/version_5/models/xlmr_lora_multitask_mean_pool_epoch8/cefr_calibration.json


In [ ]:
from pathlib import Path
import json
import os

calibration_path = (
    CHECKPOINT_DIR / "cefr_calibration.json"
)

print("Expected path:", calibration_path)
print("Exists:", calibration_path.exists())

if calibration_path.exists():
    print("Size:", calibration_path.stat().st_size, "bytes")

    with open(
        calibration_path,
        "r",
        encoding="utf-8",
    ) as file:
        saved_calibration = json.load(file)

    print(saved_calibration)
else:
    print("Files currently present:")
    for path in CHECKPOINT_DIR.iterdir():
        print("-", path.name)

os.sync()

Expected path: /content/drive/MyDrive/SHL_take_home_assessment/version_5/models/xlmr_lora_multitask_mean_pool_epoch8/cefr_calibration.json
Exists: True
Size: 654 bytes
{'method': 'adjacent_class_validation_cutpoints', 'cut_points': [0.6241558194160461, 1.572928547859192, 2.2835617065429688, 3.4686732292175293, 4.533013343811035], 'pair_accuracies': [0.875, 0.8228346456692913, 0.7709251101321586, 0.8454545454545455, 0.8813559322033898], 'class_mapping': {'0': 'A1', '1': 'A2', '2': 'B1', '3': 'B2', '4': 'C1', '5': 'C2'}, 'exact_validation_rows': 430, 'metrics': {'exact_accuracy': 0.6767441860465117, 'pearson': 0.8521710374023652, 'mean_distance': 0.3674418604651163, 'within_one_level': 0.958139534883721}}
